# Feature Engineering for Water Leak Detection

This notebook transforms the cleaned leak dataset into model-ready features.

## Objectives
- Build physically meaningful derived variables.
- Encode sensor identity for model consumption.
- Remove weak or noisy inputs.
- Visualize feature separability between normal and leak states.
- Export the engineered dataset for training and evaluation.

## Feature Strategy
1. Temporal change feature (pressure delta).
2. Sensor-normalized anomaly feature (flow z-score).
3. Pressure-to-flow interaction ratio.
4. Categorical sensor encoding.
5. Final feature validation and export.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_DIR = "../data"
FIG_DIR  = "../paper/images"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(filename, dpi=300):
    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f"Saved → {path}")

## Load Cleaned Dataset
We load the burst-filtered dataset created in Notebook 1 and ensure timestamps are parsed.

This gives a stable baseline before deriving additional features.

In [2]:
df = pd.read_csv(f"{DATA_DIR}/data_clean.csv")
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"Shape  : {df.shape}")
print(f"Leaks  : {df['Leak Status'].sum()}")
df.head()


Shape  : (5590, 6)
Leaks  : 774


,Timestamp,Sensor_ID,Pressure (bar),Flow Rate (L/s),Temperature (°C),Leak Status
0,2024-01-01 01:10:00,S001,2.595338,96.647348,14.877750,0
1,2024-01-01 02:20:00,S001,2.726076,126.229817,20.437192,0
2,2024-01-01 02:30:00,S001,3.096358,127.662703,22.565652,0
3,2024-01-01 05:15:00,S001,3.477227,81.002654,14.109417,0
4,2024-01-01 05:45:00,S001,3.257879,173.968620,14.800744,0


In [3]:
# --- Temporal split BEFORE feature engineering (train/test by time)
# `df` is expected from the previous cell (cleaned, Timestamp parsed)

df_sorted = df.sort_values('Timestamp').reset_index(drop=True)
N = len(df_sorted)
train_end = int(N * 0.80)

train_raw = df_sorted.iloc[:train_end].copy()
test_raw  = df_sorted.iloc[train_end:].copy()

print(f"Temporal split -> Train: {len(train_raw)}, Test: {len(test_raw)}")

# Compute per-sensor flow statistics ON THE TRAINING SET only
sensor_stats = (
    train_raw
    .groupby('Sensor_ID')['Flow Rate (L/s)']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'mu', 'std': 'sigma'})
)

# Stabilize small std and fill NaNs
sensor_stats['sigma'] = sensor_stats['sigma'].fillna(sensor_stats['sigma'].mean())
sensor_stats['sigma'] = sensor_stats['sigma'].abs() + 1e-8

# Consistent sensor encoding based on training split
sensor_map = {s: i for i, s in enumerate(sorted(train_raw['Sensor_ID'].unique()))}


def engineer(df_part, sensor_stats, sensor_map):
    dfp = df_part.copy()
    dfp = dfp.sort_values(['Sensor_ID', 'Timestamp']).reset_index(drop=True)

    # 1) Pressure delta per sensor (local temporal change)
    dfp['pressure_delta'] = dfp.groupby('Sensor_ID')['Pressure (bar)'].diff().fillna(0)

    # 2) PF ratio
    dfp['pf_ratio'] = dfp['Pressure (bar)'] / (dfp['Flow Rate (L/s)'] + 1e-8)

    # 3) Flow z-score using TRAINING sensor statistics
    dfp = dfp.merge(sensor_stats, left_on='Sensor_ID', right_index=True, how='left')
    global_mu = sensor_stats['mu'].mean()
    global_sigma = sensor_stats['sigma'].mean()
    dfp['mu'] = dfp['mu'].fillna(global_mu)
    dfp['sigma'] = dfp['sigma'].fillna(global_sigma)
    dfp['flow_zscore'] = (dfp['Flow Rate (L/s)'] - dfp['mu']) / dfp['sigma']
    dfp = dfp.drop(columns=['mu', 'sigma'])

    # 4) Sensor encoding (train-based mapping; unseen sensors get -1)
    dfp['sensor_enc'] = dfp['Sensor_ID'].map(sensor_map).fillna(-1).astype(int)

    return dfp


# Engineer each split
train_eng = engineer(train_raw, sensor_stats, sensor_map)
test_eng  = engineer(test_raw, sensor_stats, sensor_map)

# Save engineered splits
train_path = f"{DATA_DIR}/train_feat.csv"
test_path  = f"{DATA_DIR}/test_feat.csv"

train_eng.to_csv(train_path, index=False)
test_eng.to_csv(test_path, index=False)

# Save merged engineered table for bookkeeping
df_engineered = pd.concat([train_eng, test_eng], axis=0).sort_values('Timestamp').reset_index(drop=True)
df_engineered.to_csv(f"{DATA_DIR}/data_engineered.csv", index=False)

print(f"Saved -> {train_path}  (n={len(train_eng)})")
print(f"Saved -> {test_path}   (n={len(test_eng)})")
print(f"Leak rates - Train: {train_eng['Leak Status'].mean()*100:.3f}%, Test: {test_eng['Leak Status'].mean()*100:.3f}%")

Temporal split -> Train: 4472, Test: 1118
Saved -> ../data/train_feat.csv  (n=4472)
Saved -> ../data/test_feat.csv   (n=1118)
Leak rates - Train: 12.701%, Test: 18.426%


## Feature 1: Pressure Delta
Pressure delta measures how pressure changes between consecutive readings for the same sensor.

Sudden negative shifts can be informative during leak onset.

## Feature 2: Flow Z-Score per Sensor
Raw flow values differ by sensor baseline, so this feature standardizes flow within each sensor.

A high absolute z-score indicates behavior that is unusual for that specific sensor.

## Feature 3: Pressure-to-Flow Ratio
This interaction feature combines pressure and flow into a single hydraulic consistency indicator.

Leak scenarios often reduce pressure while increasing flow, which can push this ratio downward.

## Sensor Encoding
Sensor identifiers are categorical values, so we map them to integer codes for downstream models.

This preserves sensor identity while keeping the table fully numeric.

## Remove Low-Value Feature
Temperature is dropped based on prior EDA evidence showing minimal correlation with leak status.

Removing weak signals can reduce noise and simplify modeling.

## Final Feature Set Definition
We explicitly list the modeling features and inspect their summary statistics.

This acts as a checkpoint before plotting and exporting the engineered table.

## Distribution Comparison: Normal vs Leak
These histograms check whether engineered features separate leak and non-leak classes.

Visible class separation suggests stronger predictive utility for supervised learning.

## Export Engineered Dataset
The engineered DataFrame is saved as data_engineered.csv for model training and evaluation notebooks.

The printed summary confirms row count, leak count, and final saved features.

## Feature Engineering Takeaways
- Derived hydraulic features capture dynamics beyond raw sensor values.
- Sensor-aware normalization and encoding improve cross-sensor comparability.
- The exported engineered dataset is ready for model development in the next notebook.